In [ ]:
# Cell 0: Setup & Imports
import numpy as np
import pandas as pd
import os, sys, json, pickle, math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from pathlib import Path
import cv2
from PIL import Image
from itertools import chain
from einops import rearrange
from sklearn.decomposition import PCA

!pip install -q wandb --upgrade
import wandb
print('Imports OK')

In [ ]:
# Cell 1: Git Clone & Setup
REPO_URL = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH = 'origin'

if not os.path.exists(REPO_NAME):
    print(f'Cloning {REPO_URL}...')
    !git clone {REPO_URL} -b {BRANCH}
else:
    print('Repo already exists.')

if os.path.basename(os.getcwd()) != REPO_NAME:
    target_dir = os.path.join(os.getcwd(), REPO_NAME, 'causalvid')
    if os.path.exists(target_dir):
        os.chdir(target_dir)
    elif os.path.exists(REPO_NAME):
        os.chdir(REPO_NAME)
    print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 2: Merge DINOv3 Features (train/test/valid -> merged folder)
import shutil
from tqdm.auto import tqdm

CLIP_SPLIT_PATH = '/kaggle/input/datasets/danielq07/dinov3-feat/features'
CLIP_MERGED_PATH = '/kaggle/working/dinov2_T16_dim1024_merge'

print(f'\nSource: {CLIP_SPLIT_PATH}')
if os.path.exists(CLIP_SPLIT_PATH):
    subfolders = os.listdir(CLIP_SPLIT_PATH)
    print(f'Subfolders found: {subfolders}')
else:
    print('Source path not found!')
    subfolders = []

os.makedirs(CLIP_MERGED_PATH, exist_ok=True)

total_copied = 0
for split in ['train', 'test', 'valid', 'val']:
    split_folder = os.path.join(CLIP_SPLIT_PATH, split)
    if not os.path.exists(split_folder):
        continue

    pt_files = [f for f in os.listdir(split_folder) if f.endswith('.pt')]
    print(f'\n{split}: {len(pt_files)} files')

    for fname in tqdm(pt_files, desc=f'Copying {split}'):
        src = os.path.join(split_folder, fname)
        dst = os.path.join(CLIP_MERGED_PATH, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            total_copied += 1

final_count = len([f for f in os.listdir(CLIP_MERGED_PATH) if f.endswith('.pt')])
print(f'\nMerge complete! Total .pt files: {final_count}')
print(f'Merged path: {CLIP_MERGED_PATH}')

In [ ]:
# Cell 3: Configuration
print('=== Configuration ===')

WANDB_API_KEY = ''
WANDB_PROJECT = 'transtr-causalvid-dino'
WANDB_ARTIFACT = 'best-model-dinov3-ncod-hum:latest'

VIT_FEATURE_PATH = CLIP_MERGED_PATH
OBJ_FEATURE_PATH = '/kaggle/input/object-detection-causal-full'
ANNOTATION_PATH  = '/kaggle/input/text-annotation/QA'
SPLIT_DIR        = '/kaggle/input/casual-vid-data-split/split'
VIDEO_ROOT       = '/kaggle/input/raw-casual/root/Desktop/Data/test'

ALL_QUESTION_TYPES = [
    'descriptive', 'explanatory', 'predictive',
    'predictive_reason', 'counterfactual', 'counterfactual_reason'
]

class Config:
    topK_frame = 16
    objs = 20
    frames = 16
    select_frames = 5
    topK_obj = 12
    frame_feat_dim = 1024
    obj_feat_dim = 2053
    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    n_query = 5
    hard_eval = True

args = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Cell 3: Load Model from W&B
print('=== Load Model ===')

from networks.model import VideoQAmodel
from networks.topk import HardtopK
from utils.util import set_seed, transform_bb
set_seed(999)

wandb.login(key=WANDB_API_KEY)
api = wandb.Api()
artifact = api.artifact(f'{WANDB_PROJECT}/{WANDB_ARTIFACT}')
artifact_dir = artifact.download()
print(f'Downloaded to: {artifact_dir}')

ckpt_path = None
for f in os.listdir(artifact_dir):
    if f.endswith('.ckpt') or f.endswith('.pt') or f.endswith('.pth'):
        ckpt_path = os.path.join(artifact_dir, f)
        break
print(f'Checkpoint: {ckpt_path}')

cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames

model = VideoQAmodel(**cfg)

ckpt = torch.load(ckpt_path, map_location=device)
if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded model_state_dict (epoch {ckpt.get("epoch", "?")})')
else:
    model.load_state_dict(ckpt)
    print('Loaded raw state_dict')

model.to(device)
model.eval()
print('Model loaded!')

In [ ]:
# Cell 4: Helper Functions
print('=== Helper Functions ===')

def find_video_path(video_id, video_root):
    video_root = Path(video_root)
    for ext in ['.mp4', '.avi', '.mkv', '.mov', '.webm']:
        path = video_root / f'{video_id}{ext}'
        if path.exists():
            return path
        for subpath in video_root.rglob(f'{video_id}{ext}'):
            return subpath
    return None

def load_video_frames(video_path, num_frames=16):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    frames, frame_indices = [], []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frame_indices.append(idx)
    cap.release()
    return frames, frame_indices, total

def load_qa_data(video_id, annotation_path):
    vp = os.path.join(annotation_path, video_id)
    qa_data = {}
    text_file = os.path.join(vp, 'text.json')
    ans_file  = os.path.join(vp, 'answer.json')
    if os.path.exists(text_file) and os.path.exists(ans_file):
        with open(text_file, 'r', encoding='utf-8') as f:
            text_data = json.load(f)
        with open(ans_file, 'r', encoding='utf-8') as f:
            ans_data = json.load(f)
        for qtype in ['descriptive', 'explanatory', 'predictive', 'counterfactual']:
            if qtype in text_data and qtype in ans_data:
                qa_data[qtype] = {
                    'question': text_data[qtype].get('question', ''),
                    'choices': text_data[qtype].get('answer', []),
                    'correct_idx': ans_data[qtype].get('answer', 0)
                }
                if 'reason' in text_data[qtype]:
                    qa_data[f'{qtype}_reason'] = {
                        'question': 'Why?',
                        'choices': text_data[qtype].get('reason', []),
                        'correct_idx': ans_data[qtype].get('reason', 0)
                    }
    return qa_data

def load_features(video_id):
    """Load ViT frame features and object features for a video."""
    ff = torch.load(os.path.join(VIT_FEATURE_PATH, f'{video_id}.pt'), weights_only=True)
    if isinstance(ff, np.ndarray):
        ff = torch.from_numpy(ff)
    ff = ff.float()
    nf = ff.shape[0]
    if nf > args.topK_frame:
        indices = np.linspace(0, nf - 1, args.topK_frame).astype(int)
        ff = ff[indices]
    elif nf < args.topK_frame:
        ff = torch.cat([ff, torch.zeros(args.topK_frame - nf, ff.shape[1])], 0)

    obj_pkl = None
    for subdir in os.listdir(OBJ_FEATURE_PATH):
        subdir_path = os.path.join(OBJ_FEATURE_PATH, subdir)
        if os.path.isdir(subdir_path):
            pkl_path = os.path.join(subdir_path, f'{video_id}.pkl')
            if os.path.exists(pkl_path):
                obj_pkl = pkl_path
                break

    objs, obj_bboxes = [], []
    if obj_pkl:
        with open(obj_pkl, 'rb') as f:
            data = pickle.load(f)
        feats = np.array(data.get('features'))
        bboxes = np.array(data.get('bboxes'))
        num_frames = feats.shape[0]
        obj_indices = (np.linspace(0, num_frames - 1, args.topK_frame).astype(int)
                       if num_frames > args.topK_frame else range(num_frames))
        for i in obj_indices:
            feat = torch.from_numpy(feats[i]).float()
            bbox = torch.from_numpy(bboxes[i]).float()
            if feat.shape[0] > args.objs:
                feat, bbox = feat[:args.objs], bbox[:args.objs]
            elif feat.shape[0] < args.objs:
                p = args.objs - feat.shape[0]
                feat = torch.cat([feat, torch.zeros(p, feat.shape[1])], 0)
                bbox = torch.cat([bbox, torch.zeros(p, bbox.shape[1])], 0)
            bb = torch.from_numpy(transform_bb(bbox.numpy(), 640, 480)).float()
            objs.append(torch.cat([feat, bb], -1))
            obj_bboxes.append(bbox.numpy())

    while len(objs) < args.topK_frame:
        objs.append(torch.zeros(args.objs, 2053))
        obj_bboxes.append(np.zeros((args.objs, 4)))

    of = torch.stack(objs)
    return ff, of, obj_bboxes

print('Helper functions defined')

In [ ]:
# Cell 5: DINOv3 Feature Heatmaps from Pre-Extracted .pt Files
print('=== DINOv3 Feature Heatmap Utilities ===')

def get_dino_heatmap_from_pt(video_id, frame_idx, feature_path=None):
    """
    Build a PCA-based spatial heatmap from the pre-extracted DINOv3 .pt file.

    If the .pt stores full patch tokens [T, num_patches+1, dim]:
        -> PCA on patches -> spatial RGB heatmap
    If the .pt stores only global features [T, dim]:
        -> PCA on feature dimensions -> 1-D activation bar (reshaped to square)
    """
    if feature_path is None:
        feature_path = VIT_FEATURE_PATH
    pt_path = os.path.join(feature_path, f'{video_id}.pt')
    if not os.path.exists(pt_path):
        return None

    feat = torch.load(pt_path, weights_only=True)
    if isinstance(feat, np.ndarray):
        feat = torch.from_numpy(feat)
    feat = feat.float()

    if feat.ndim == 3:
        # [T, num_tokens, dim] — full patch-level features
        frame_feat = feat[frame_idx]             # [num_tokens, dim]
        num_tokens = frame_feat.shape[0]
        num_patches = int(math.sqrt(num_tokens - 1)) ** 2
        h = w = int(math.sqrt(num_patches))
        patch_feat = frame_feat[1:num_patches+1, :]  # skip CLS

        pca = PCA(n_components=3)
        pca_rgb = pca.fit_transform(patch_feat.numpy()).reshape(h, w, 3)
        for c in range(3):
            ch = pca_rgb[:, :, c]
            pca_rgb[:, :, c] = (ch - ch.min()) / (ch.max() - ch.min() + 1e-8)
        return Image.fromarray((pca_rgb * 255).astype(np.uint8))

    elif feat.ndim == 2:
        # [T, dim] — global features only; create activation magnitude map
        frame_feat = feat[frame_idx]  # [dim]
        side = int(math.ceil(math.sqrt(frame_feat.shape[0])))
        padded = torch.zeros(side * side)
        padded[:frame_feat.shape[0]] = frame_feat.abs()
        grid = padded.reshape(side, side).numpy()
        grid = (grid - grid.min()) / (grid.max() - grid.min() + 1e-8)
        cmap = plt.cm.viridis
        rgb = (cmap(grid)[:, :, :3] * 255).astype(np.uint8)
        return Image.fromarray(rgb)

    return None

print('DINOv3 heatmap from .pt defined')

In [ ]:
# Cell 6: forward_with_full_analysis() — Deep Layer-by-Layer Inference
print('=== Defining deep analysis forward pass ===')

def _run_decoder_layers(decoder, tgt, memory,
                        tgt_mask=None, memory_mask=None,
                        tgt_key_padding_mask=None,
                        memory_key_padding_mask=None,
                        pos=None, query_pos=None):
    """Manually iterate decoder layers, collect per-layer cross-attention."""
    output = tgt
    layer_attns = []
    for layer in decoder.layers:
        output, c_att = layer(
            output, memory,
            tgt_mask=tgt_mask, memory_mask=memory_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
            pos=pos, query_pos=query_pos
        )
        layer_attns.append(c_att.detach().cpu())
    if decoder.norm is not None:
        output = decoder.norm(output)
    return output, layer_attns


def _run_encoder_layers_with_attn(encoder, src,
                                   mask=None,
                                   src_key_padding_mask=None,
                                   pos=None):
    """Manually iterate encoder layers, hooking self-attention to capture weights."""
    output = src
    layer_attns = []

    for layer in encoder.layers:
        q = k = layer.with_pos_embed(output, pos)
        src2, self_att = layer.self_attn(
            q, k, value=output,
            attn_mask=mask,
            key_padding_mask=src_key_padding_mask,
            output_attentions=True
        )
        layer_attns.append(self_att.detach().cpu())
        output = output + layer.dropout1(src2)
        output = layer.norm1(output)
        src2 = layer.linear2(layer.dropout(layer.activation(layer.linear1(output))))
        output = output + layer.dropout2(src2)
        output = layer.norm2(output)

    if encoder.norm is not None:
        output = encoder.norm(output)
    return output, layer_attns


@torch.no_grad()
def forward_with_full_analysis(model, video_id, qtype, device):
    """
    Run the full VideoQAmodel forward pass step-by-step,
    collecting per-layer attention weights from every transformer stage.
    """
    ff, of, obj_bboxes = load_features(video_id)
    qa_data = load_qa_data(video_id, ANNOTATION_PATH)
    if qtype not in qa_data:
        raise ValueError(f'{qtype} not found for {video_id}')
    qa = qa_data[qtype]
    qns, choices, correct_idx = qa['question'], qa['choices'], qa['correct_idx']
    ans_strings = [f'{qns} [SEP] {c}' for c in choices]

    ff = ff.unsqueeze(0).to(device)
    of = of.unsqueeze(0).to(device)
    B, F_total, O = of.size()[:3]
    result = {}

    # --- Tokenize question for axis labels ---
    tokenized = model.tokenizer(qns, return_tensors='pt')
    q_token_labels = model.tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0])
    result['q_token_labels'] = q_token_labels

    # ========================================================
    # Stage 1: Frame Resize + Text Encoding
    # ========================================================
    frame_feat = model.frame_resize(ff)  # [B, F, d_model]
    q_local, q_mask = model.forward_text([qns], device)
    result['frame_feat_norms'] = ff[0].norm(dim=-1).cpu().numpy()

    # ========================================================
    # Stage 2: Frame Decoder (2 layers) — frame ← question cross-attn
    # ========================================================
    frame_mask = torch.ones(B, F_total).bool().to(device)
    frame_local, frame_layer_attns = _run_decoder_layers(
        model.frame_decoder, frame_feat, q_local,
        memory_key_padding_mask=q_mask,
        query_pos=model.pos_encoder_1d(frame_mask, model.d_model)
    )
    result['frame_decoder_attns'] = frame_layer_attns  # list of [B, F, q_len]

    frame_att_last = frame_layer_attns[-1].to(device)
    idx_frame = rearrange(
        HardtopK(frame_att_last.flatten(1, 2), model.frame_topK),
        'b (f q) k -> b f q k', f=F_total
    ).sum(-2)

    frame_importance = frame_att_last.mean(dim=-1)[0].cpu().numpy()
    topk_frame_indices = np.argsort(frame_importance)[-model.frame_topK:][::-1]
    result['frame_importance'] = frame_importance
    result['topk_frame_indices'] = np.sort(topk_frame_indices)

    frame_local = (frame_local.transpose(1, 2) @ idx_frame).transpose(1, 2)

    # ========================================================
    # Stage 3: Object Decoder (2 layers) — obj ← question cross-attn
    # ========================================================
    obj_feat = (of.flatten(-2, -1).transpose(1, 2) @ idx_frame).transpose(1, 2)
    obj_feat = obj_feat.view(B, model.frame_topK, O, -1)
    obj_local = model.obj_resize(obj_feat)

    q_local_rep = q_local.repeat_interleave(model.frame_topK, dim=0)
    q_mask_rep = q_mask.repeat_interleave(model.frame_topK, dim=0) if q_mask is not None else None

    obj_local_flat = obj_local.flatten(0, 1)
    obj_local_out, obj_layer_attns = _run_decoder_layers(
        model.obj_decoder, obj_local_flat, q_local_rep,
        memory_key_padding_mask=q_mask_rep
    )
    result['obj_decoder_attns'] = obj_layer_attns  # list of [B*topK_frame, O, q_len]

    obj_att_last = obj_layer_attns[-1].to(device)
    idx_obj = rearrange(
        HardtopK(obj_att_last.flatten(1, 2), model.obj_topK),
        'b (o q) k -> b o q k', o=O
    ).sum(-2)

    obj_importance_per_frame = obj_att_last.mean(dim=-1).cpu().numpy()  # [topK_frame, O]
    topk_obj_per_frame = np.argsort(obj_importance_per_frame, axis=-1)[:, -model.obj_topK:]
    result['obj_importance_per_frame'] = obj_importance_per_frame
    result['topk_obj_per_frame'] = topk_obj_per_frame

    obj_local_out = (obj_local_out.transpose(1, 2) @ idx_obj).transpose(1, 2)
    obj_local_out = obj_local_out.view(B, model.frame_topK, model.obj_topK, -1)

    # ========================================================
    # Stage 4: Frame-Object Decoder (2 layers) — frame ← objects cross-attn
    # ========================================================
    fo_local, fo_layer_attns = _run_decoder_layers(
        model.fo_decoder, frame_local, obj_local_out.flatten(1, 2)
    )
    result['fo_decoder_attns'] = fo_layer_attns  # list of [B, topK_frame, topK_frame*obj_topK]

    # ========================================================
    # Stage 5: VL Encoder (2 layers) — self-attention on [frame_obj, q_local]
    # ========================================================
    frame_mask_sel = torch.ones(B, model.frame_topK).bool().to(device)
    fo_local = fo_local.view(B, model.frame_topK, -1)
    frame_qns_mask = torch.cat((frame_mask_sel, q_mask), dim=1).bool()
    vl_input = torch.cat((fo_local, q_local), dim=1)
    vl_pos = model.pos_encoder_1d(frame_qns_mask.bool(), model.d_model)

    mem, vl_layer_attns = _run_encoder_layers_with_attn(
        model.vl_encoder, vl_input,
        src_key_padding_mask=frame_qns_mask,
        pos=vl_pos
    )
    result['vl_encoder_attns'] = vl_layer_attns  # list of [B, seq, seq]
    result['vl_num_frame_tokens'] = model.frame_topK

    # ========================================================
    # Stage 6: Answer Decoder (2 layers) — answers ← memory cross-attn
    # ========================================================
    a_seq, _ = model.forward_text(list(chain(*[ans_strings])), device, has_ans=True)
    a_seq = rearrange(a_seq, '(n b) t c -> b n t c', b=B)
    tgt = a_seq[:, :, 0, :]  # [CLS] tokens: [B, 5, d_model]

    ans_out, ans_layer_attns = _run_decoder_layers(
        model.ans_decoder, tgt, mem,
        memory_key_padding_mask=frame_qns_mask
    )
    result['ans_decoder_attns'] = ans_layer_attns  # list of [B, 5, mem_len]

    # ========================================================
    # Final prediction
    # ========================================================
    out = model.classifier(ans_out).squeeze(-1)
    pred = out.argmax(-1).item()
    probs = torch.softmax(out, dim=-1)[0].cpu().numpy()

    result['prediction'] = pred
    result['probs'] = probs
    result['correct_idx'] = correct_idx
    result['question'] = qns
    result['choices'] = choices
    result['obj_bboxes'] = obj_bboxes
    result['qtype'] = qtype
    result['video_id'] = video_id

    return result

print('Deep analysis forward pass defined')

In [ ]:
# Cell 7: Visualization Functions
print('=== Defining visualization functions ===')

# ----------------------------------------------------------------
# Plot 1: Frame Selection + DINOv3 Spatial Heatmaps
# ----------------------------------------------------------------
def plot_frame_selection(result, frames, dino_heatmaps=None):
    topk = result['topk_frame_indices']
    importance = result['frame_importance']
    frame_attns = result['frame_decoder_attns']
    q_tokens = result['q_token_labels']
    norms = result['frame_feat_norms']
    n_layers = len(frame_attns)
    F_total = len(importance)

    has_dino = dino_heatmaps is not None and len(dino_heatmaps) > 0
    n_rows = 2 + n_layers + (1 if has_dino else 0)
    fig = plt.figure(figsize=(24, 4 * n_rows))
    fig.suptitle(
        f'Frame Selection Analysis \u2014 {result["video_id"]} \u2014 {result["qtype"]}',
        fontsize=16, fontweight='bold', y=0.99
    )
    gs = fig.add_gridspec(n_rows, max(F_total, len(topk)), hspace=0.4)

    row = 0
    # Row 1: All 16 frames
    for i in range(F_total):
        ax = fig.add_subplot(gs[row, i])
        if i < len(frames):
            ax.imshow(frames[i])
        is_selected = i in topk
        for spine in ax.spines.values():
            spine.set_color('lime' if is_selected else 'gray')
            spine.set_linewidth(4 if is_selected else 1)
        ax.set_title(f'F{i}\n{importance[i]:.3f}',
                     fontsize=8, color='green' if is_selected else 'gray')
        ax.set_xticks([]); ax.set_yticks([])

    # Row 2 (optional): DINOv3 PCA heatmaps for selected frames
    if has_dino:
        row += 1
        n_sel = len(topk)
        cols_per = max(1, F_total // n_sel)
        for j, fidx in enumerate(topk):
            ax = fig.add_subplot(gs[row, j * cols_per:(j + 1) * cols_per])
            if fidx < len(frames) and j < len(dino_heatmaps):
                orig = Image.fromarray(frames[fidx]).resize(dino_heatmaps[j].size)
                blended = Image.blend(orig, dino_heatmaps[j], alpha=0.5)
                ax.imshow(blended)
            ax.set_title(f'DINOv3 PCA \u2014 Frame {fidx}', fontsize=9)
            ax.axis('off')

    # Rows 3+: Attention heatmaps per layer
    for li in range(n_layers):
        row += 1
        ax = fig.add_subplot(gs[row, :])
        att = frame_attns[li][0].numpy()  # [F, q_len]
        im = ax.imshow(att, aspect='auto', cmap='viridis')
        ax.set_xlabel('Question Tokens')
        ax.set_ylabel('Frame Index')
        ax.set_title(f'Frame Decoder Layer {li} Cross-Attention', fontsize=11)
        ax.set_xticks(range(len(q_tokens)))
        ax.set_xticklabels(q_tokens, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(F_total))
        ax.set_yticklabels([f'F{i}{" *" if i in topk else ""}' for i in range(F_total)], fontsize=8)
        plt.colorbar(im, ax=ax, fraction=0.02)

    # Last row: Bar chart of importance + feature norms
    row += 1
    ax = fig.add_subplot(gs[row, :])
    x = np.arange(F_total)
    colors = ['lime' if i in topk else 'lightgray' for i in range(F_total)]
    bars = ax.bar(x - 0.2, importance, 0.4, color=colors, edgecolor='black', label='Attn Score')
    norm_vals = norms / (norms.max() + 1e-8) * importance.max()
    ax.bar(x + 0.2, norm_vals, 0.4, color='steelblue', alpha=0.5, label='Feature Norm (scaled)')
    ax.set_xlabel('Frame'); ax.set_ylabel('Score')
    ax.set_title('Frame Importance (attention mean) vs DINOv3 Feature Norm')
    ax.legend()
    plt.tight_layout()
    plt.show()


# ----------------------------------------------------------------
# Plot 2: Object Selection & Spatial Rationalization
# ----------------------------------------------------------------
def plot_object_selection(result, frames, dino_heatmaps=None):
    topk_frames = result['topk_frame_indices']
    obj_importance = result['obj_importance_per_frame']  # [topK_frame, O]
    topk_objs = result['topk_obj_per_frame']            # [topK_frame, obj_topK]
    obj_bboxes = result['obj_bboxes']
    obj_attns = result['obj_decoder_attns']
    q_tokens = result['q_token_labels']
    n_sel = len(topk_frames)

    has_dino = dino_heatmaps is not None and len(dino_heatmaps) > 0
    n_cols = 2 if has_dino else 1
    n_layers = len(obj_attns)

    fig, axes = plt.subplots(n_sel, n_cols + n_layers, figsize=(6 * (n_cols + n_layers), 4 * n_sel))
    if n_sel == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(
        f'Object Selection \u2014 {result["video_id"]} \u2014 {result["qtype"]}',
        fontsize=14, fontweight='bold'
    )

    cmap = plt.cm.Reds
    for j, fidx in enumerate(topk_frames):
        col = 0
        # Column 1: frame + bounding boxes
        ax = axes[j, col]
        if fidx < len(frames):
            ax.imshow(frames[fidx])
        obj_scores = obj_importance[j]
        norm = Normalize(vmin=obj_scores.min(), vmax=obj_scores.max() + 1e-8)
        selected_set = set(topk_objs[j])
        if fidx < len(obj_bboxes):
            for oi in range(len(obj_bboxes[fidx])):
                if obj_scores[oi] < 1e-6:
                    continue
                x1, y1, x2, y2 = obj_bboxes[fidx][oi][:4]
                is_sel = oi in selected_set
                color = cmap(norm(obj_scores[oi]))
                ax.add_patch(plt.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    fill=False, edgecolor=color,
                    linewidth=3 if is_sel else 1,
                    linestyle='-' if is_sel else '--'
                ))
                if is_sel:
                    ax.text(x1, y1 - 2, f'O{oi}', fontsize=7, color='white',
                            bbox=dict(facecolor=color, alpha=0.7, pad=1))
        ax.set_title(f'Frame {fidx} \u2014 Objects', fontsize=10)
        ax.axis('off')

        # Column 2 (optional): DINOv3 heatmap side-by-side
        if has_dino:
            col += 1
            ax = axes[j, col]
            if fidx < len(frames) and j < len(dino_heatmaps):
                orig = Image.fromarray(frames[fidx]).resize(dino_heatmaps[j].size)
                blended = Image.blend(orig, dino_heatmaps[j], alpha=0.5)
                ax.imshow(blended)
            ax.set_title(f'DINOv3 PCA \u2014 Frame {fidx}', fontsize=10)
            ax.axis('off')

        # Remaining columns: obj decoder attention heatmaps per layer
        for li in range(n_layers):
            col += 1
            ax = axes[j, col]
            att = obj_attns[li][j].numpy()  # [O, q_len]
            im = ax.imshow(att, aspect='auto', cmap='magma')
            ax.set_xlabel('Q Tokens', fontsize=8)
            ax.set_ylabel('Object Idx', fontsize=8)
            sel_labels = [f'O{i}{" *" if i in selected_set else ""}'
                          for i in range(att.shape[0])]
            ax.set_yticks(range(min(20, att.shape[0])))
            ax.set_yticklabels(sel_labels[:20], fontsize=6)
            ax.set_xticks(range(len(q_tokens)))
            ax.set_xticklabels(q_tokens, rotation=45, ha='right', fontsize=6)
            ax.set_title(f'Obj Dec L{li} \u2014 Frame {fidx}', fontsize=9)
            plt.colorbar(im, ax=ax, fraction=0.03)

    plt.tight_layout()
    plt.show()


# ----------------------------------------------------------------
# Plot 3: Frame-Object Fusion (fo_decoder cross-attention)
# ----------------------------------------------------------------
def plot_fo_fusion(result):
    fo_attns = result['fo_decoder_attns']
    topk_frames = result['topk_frame_indices']
    n_layers = len(fo_attns)

    fig, axes = plt.subplots(1, n_layers, figsize=(8 * n_layers, 5))
    if n_layers == 1:
        axes = [axes]
    fig.suptitle(
        f'Frame-Object Fusion \u2014 {result["video_id"]} \u2014 {result["qtype"]}',
        fontsize=14, fontweight='bold'
    )

    for li in range(n_layers):
        ax = axes[li]
        att = fo_attns[li][0].numpy()  # [topK_frame, topK_frame * obj_topK]
        im = ax.imshow(att, aspect='auto', cmap='inferno')
        ax.set_ylabel('Frame Token')
        ax.set_xlabel('Object Tokens (flattened)')
        ax.set_yticks(range(att.shape[0]))
        ax.set_yticklabels([f'F{topk_frames[i]}' for i in range(att.shape[0])], fontsize=9)
        ax.set_title(f'fo_decoder Layer {li}', fontsize=12)
        plt.colorbar(im, ax=ax, fraction=0.03)

    plt.tight_layout()
    plt.show()


# ----------------------------------------------------------------
# Plot 4: VL Encoder Self-Attention (Video-Text Interaction)
# ----------------------------------------------------------------
def plot_vl_encoder(result):
    vl_attns = result['vl_encoder_attns']
    n_frame_tokens = result['vl_num_frame_tokens']
    q_tokens = result['q_token_labels']
    topk_frames = result['topk_frame_indices']
    n_layers = len(vl_attns)

    token_labels = [f'F{topk_frames[i]}' for i in range(n_frame_tokens)] + list(q_tokens)

    fig, axes = plt.subplots(1, n_layers, figsize=(10 * n_layers, 8))
    if n_layers == 1:
        axes = [axes]
    fig.suptitle(
        f'VL Encoder Self-Attention (Video \u2194 Text) \u2014 {result["video_id"]} \u2014 {result["qtype"]}',
        fontsize=14, fontweight='bold'
    )

    for li in range(n_layers):
        ax = axes[li]
        att = vl_attns[li][0].numpy()  # [seq, seq]
        seq_len = att.shape[0]
        labels = token_labels[:seq_len]
        im = ax.imshow(att, cmap='Blues')
        ax.set_xticks(range(seq_len))
        ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=7)
        ax.set_yticks(range(seq_len))
        ax.set_yticklabels(labels, fontsize=7)
        ax.set_title(f'vl_encoder Layer {li}', fontsize=12)

        # Draw separator line between frame and text tokens
        ax.axhline(y=n_frame_tokens - 0.5, color='red', linewidth=1.5, linestyle='--')
        ax.axvline(x=n_frame_tokens - 0.5, color='red', linewidth=1.5, linestyle='--')
        ax.text(-1.5, n_frame_tokens / 2, 'Video', fontsize=8, ha='center', va='center',
                rotation=90, color='red', fontweight='bold')
        ax.text(-1.5, n_frame_tokens + (seq_len - n_frame_tokens) / 2, 'Text',
                fontsize=8, ha='center', va='center', rotation=90, color='blue', fontweight='bold')
        plt.colorbar(im, ax=ax, fraction=0.03)

    plt.tight_layout()
    plt.show()


# ----------------------------------------------------------------
# Plot 5: Answer Decoder Cross-Attention
# ----------------------------------------------------------------
def plot_answer_decoder(result):
    ans_attns = result['ans_decoder_attns']
    n_frame_tokens = result['vl_num_frame_tokens']
    q_tokens = result['q_token_labels']
    topk_frames = result['topk_frame_indices']
    choices = result['choices']
    pred = result['prediction']
    correct = result['correct_idx']
    n_layers = len(ans_attns)

    mem_labels = [f'F{topk_frames[i]}' for i in range(n_frame_tokens)] + list(q_tokens)
    ans_labels = [f'A{i}: {c[:40]}' for i, c in enumerate(choices)]

    fig, axes = plt.subplots(1, n_layers, figsize=(12 * n_layers, 5))
    if n_layers == 1:
        axes = [axes]
    fig.suptitle(
        f'Answer Decoder Cross-Attention \u2014 {result["video_id"]} \u2014 {result["qtype"]}',
        fontsize=14, fontweight='bold'
    )

    for li in range(n_layers):
        ax = axes[li]
        att = ans_attns[li][0].numpy()  # [5, mem_len]
        mem_len = att.shape[1]
        m_labels = mem_labels[:mem_len]
        im = ax.imshow(att, aspect='auto', cmap='YlOrRd')
        ax.set_xticks(range(mem_len))
        ax.set_xticklabels(m_labels, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(5))
        ax.set_yticklabels(ans_labels, fontsize=8)

        # Highlight predicted and correct rows
        for spine_side in ['left', 'right']:
            ax.spines[spine_side].set_visible(True)
        ax.axhline(y=pred - 0.5, color='blue', linewidth=2, linestyle='-')
        ax.axhline(y=pred + 0.5, color='blue', linewidth=2, linestyle='-')
        if correct != pred:
            ax.axhline(y=correct - 0.5, color='green', linewidth=2, linestyle='--')
            ax.axhline(y=correct + 0.5, color='green', linewidth=2, linestyle='--')

        # Annotate top-attended memory tokens for predicted answer
        pred_att = att[pred]
        top3_mem = np.argsort(pred_att)[-3:]
        for tidx in top3_mem:
            ax.annotate('', xy=(tidx, pred), xytext=(tidx, pred - 0.4),
                        arrowprops=dict(arrowstyle='->', color='cyan', lw=1.5))

        ax.axvline(x=n_frame_tokens - 0.5, color='red', linewidth=1, linestyle='--')
        ax.set_title(f'ans_decoder Layer {li}', fontsize=12)
        plt.colorbar(im, ax=ax, fraction=0.02)

    pred_label = 'PRED' + (' (correct)' if pred == correct else ' (WRONG)')
    legend_elems = [
        mpatches.Patch(facecolor='none', edgecolor='blue', linewidth=2, label=pred_label),
    ]
    if correct != pred:
        legend_elems.append(
            mpatches.Patch(facecolor='none', edgecolor='green', linewidth=2,
                           linestyle='--', label='GT')
        )
    axes[-1].legend(handles=legend_elems, loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.show()


# ----------------------------------------------------------------
# Plot 6: Summary Panel
# ----------------------------------------------------------------
def plot_summary(result):
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.axis('off')

    pred = result['prediction']
    correct = result['correct_idx']
    probs = result['probs']
    is_correct = pred == correct

    text_lines = []
    text_lines.append(f'Video: {result["video_id"]}    Question Type: {result["qtype"]}')
    text_lines.append(f'Question: {result["question"]}')
    text_lines.append('')
    for i, c in enumerate(result['choices']):
        marker = ''
        if i == correct:
            marker += '[GT] '
        if i == pred:
            marker += '>>> '
        text_lines.append(f'  {marker}({i}) {c}   [{probs[i]*100:.1f}%]')
    text_lines.append('')
    text_lines.append(f'Prediction: {pred} | GT: {correct} | {"CORRECT" if is_correct else "WRONG"}')
    text_lines.append('')

    topk_f = result['topk_frame_indices']
    text_lines.append(f'Selected Frames: {list(topk_f)}')

    ans_att_last = result['ans_decoder_attns'][-1][0].numpy()  # [5, mem_len]
    n_ft = result['vl_num_frame_tokens']
    pred_att = ans_att_last[pred]
    frame_att_sum = pred_att[:n_ft]
    text_att_sum = pred_att[n_ft:]
    top_frame = np.argmax(frame_att_sum)
    top_text = np.argmax(text_att_sum)
    q_tokens = result['q_token_labels']

    text_lines.append(
        f'Predicted answer attends most to: Frame F{topk_f[top_frame]} '
        f'and token \"{q_tokens[top_text]}\"'
    )

    full_text = '\n'.join(text_lines)
    ax.text(0.05, 0.5, full_text, transform=ax.transAxes, fontsize=11,
            verticalalignment='center', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightgreen' if is_correct else 'lightcoral',
                      alpha=0.4))
    ax.set_title('Inference Summary', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ----------------------------------------------------------------
# Master visualization function
# ----------------------------------------------------------------
def plot_full_analysis(result, frames, dino_heatmaps=None):
    """Run all 6 plots for a single question analysis."""
    plot_frame_selection(result, frames, dino_heatmaps)
    plot_object_selection(result, frames, dino_heatmaps)
    plot_fo_fusion(result)
    plot_vl_encoder(result)
    plot_answer_decoder(result)
    plot_summary(result)

print('Visualization functions defined')

In [ ]:
# Cell 8: Run Deep Analysis on a Single Video
print('=== Deep Analysis ===')

VIDEO_ID = 'vgrr-N5x7kE_000035_000045'  # <-- change to your video
QTYPES_TO_ANALYZE = ['descriptive', 'counterfactual', 'counterfactual_reason']

video_path = find_video_path(VIDEO_ID, VIDEO_ROOT)
if video_path is None:
    print(f'Video file not found: {VIDEO_ID}')
else:
    frames, frame_indices, total_frames = load_video_frames(video_path, args.topK_frame)
    print(f'Loaded {len(frames)} frames from {total_frames} total')

    for qtype in QTYPES_TO_ANALYZE:
        print(f'\n{"="*60}')
        print(f'Analyzing: {VIDEO_ID} | {qtype}')
        print('='*60)
        try:
            result = forward_with_full_analysis(model, VIDEO_ID, qtype, device)

            selected_indices = result['topk_frame_indices']
            dino_heatmaps = [get_dino_heatmap_from_pt(VIDEO_ID, fidx) for fidx in selected_indices]

            plot_full_analysis(result, frames, dino_heatmaps)
        except Exception as e:
            import traceback
            print(f'Error: {e}')
            traceback.print_exc()

---
## Batch Evaluation on Test Set

Standard inference on the full test set, exporting per-qtype accuracy and CSV results.

In [ ]:
# Cell 9: Batch Inference Function (lightweight, no layer analysis)
print('=== Batch Inference Setup ===')

@torch.no_grad()
def batch_inference(model, video_id, qtype, device):
    """Fast inference without layer analysis — for batch evaluation."""
    ff, of, obj_bboxes = load_features(video_id)
    qa_data = load_qa_data(video_id, ANNOTATION_PATH)
    if qtype not in qa_data:
        return None
    qa = qa_data[qtype]
    qns, choices, correct_idx = qa['question'], qa['choices'], qa['correct_idx']
    ans_strings = [f'{qns} [SEP] {c}' for c in choices]

    ff = ff.unsqueeze(0).to(device)
    of = of.unsqueeze(0).to(device)
    ans_word_batch = [tuple(ans_strings)]
    out = model(ff, of, [qns], ans_word_batch)
    pred = out.argmax(-1).item()
    probs = torch.softmax(out, dim=-1)[0].cpu().numpy()

    return {
        'prediction': pred, 'probs': probs, 'correct_idx': correct_idx,
        'question': qns, 'choices': choices
    }

print('Batch inference function defined')

In [ ]:
# Cell 10: Run Batch Evaluation & Export CSV
from tqdm import tqdm
import traceback

print('=== Batch Evaluation ===')
OUTPUT_DIR = './inference_results_ncod_hum'
MAX_TEST_VIDEOS = None  # None = full test set
os.makedirs(OUTPUT_DIR, exist_ok=True)

test_pkl = os.path.join(SPLIT_DIR, 'test.pkl')
with open(test_pkl, 'rb') as f:
    test_vids = list(pickle.load(f))
print(f'Loaded {len(test_vids)} test videos')

if MAX_TEST_VIDEOS:
    test_vids = test_vids[:MAX_TEST_VIDEOS]

all_records = []
type_stats = {q: {'correct': 0, 'total': 0} for q in ALL_QUESTION_TYPES}
errors = []

for vid in tqdm(test_vids, desc='Inference'):
    for qtype in ALL_QUESTION_TYPES:
        try:
            r = batch_inference(model, vid, qtype, device)
            if r is None:
                continue
            is_correct = r['prediction'] == r['correct_idx']
            type_stats[qtype]['total'] += 1
            type_stats[qtype]['correct'] += int(is_correct)

            record = {
                'video_id': vid, 'question_type': qtype,
                'question': r['question'],
                'correct_idx': r['correct_idx'],
                'predicted_idx': r['prediction'],
                'is_correct': is_correct,
                'confidence': r['probs'][r['prediction']],
            }
            for i, c in enumerate(r['choices']):
                record[f'a{i}'] = c
                record[f'prob_a{i}'] = r['probs'][i]
            if r['choices']:
                record['predicted_answer'] = r['choices'][r['prediction']]
                record['correct_answer'] = r['choices'][r['correct_idx']]
            all_records.append(record)
        except Exception as e:
            errors.append({'video_id': vid, 'qtype': qtype, 'error': str(e)})

# Print summary
print(f'\nInference complete: {len(all_records)} predictions, {len(errors)} errors\n')
print(f'{"QUESTION TYPE":25} | {"CORRECT":>8} | {"TOTAL":>8} | {"ACC":>8}')
print('-' * 60)
tc, tt = 0, 0
for q in ALL_QUESTION_TYPES:
    c, t = type_stats[q]['correct'], type_stats[q]['total']
    tc += c; tt += t
    print(f'{q:25} | {c:8} | {t:8} | {c/t*100:7.1f}%' if t else f'{q:25} | -- ')
print('-' * 60)
print(f'{"OVERALL":25} | {tc:8} | {tt:8} | {tc/tt*100:7.1f}%' if tt else 'No data')

# Save CSVs
df = pd.DataFrame(all_records)
df_correct = df[df['is_correct'] == True]
df_incorrect = df[df['is_correct'] == False]

df_correct.to_csv(os.path.join(OUTPUT_DIR, 'results_correct_all.csv'), index=False, encoding='utf-8-sig')
df_incorrect.to_csv(os.path.join(OUTPUT_DIR, 'results_incorrect_all.csv'), index=False, encoding='utf-8-sig')

for qtype in ALL_QUESTION_TYPES:
    df_q = df[df['question_type'] == qtype]
    df_q[df_q['is_correct']].to_csv(
        os.path.join(OUTPUT_DIR, f'results_correct_{qtype}.csv'), index=False, encoding='utf-8-sig')
    df_q[~df_q['is_correct']].to_csv(
        os.path.join(OUTPUT_DIR, f'results_incorrect_{qtype}.csv'), index=False, encoding='utf-8-sig')

if errors:
    pd.DataFrame(errors).to_csv(
        os.path.join(OUTPUT_DIR, 'inference_errors.csv'), index=False, encoding='utf-8-sig')

print(f'\nSaved to: {os.path.abspath(OUTPUT_DIR)}')
print(f'  Correct:   {len(df_correct)} -> results_correct_all.csv')
print(f'  Incorrect: {len(df_incorrect)} -> results_incorrect_all.csv')

In [ ]:
# Cell 11: Paired Metrics (PAR, CAR)
print('=== Paired Metrics ===')

df = pd.DataFrame(all_records)

def compute_paired_accuracy(df, base_type, reason_type):
    """Both base and reason must be correct for the pair to count."""
    base = df[df['question_type'] == base_type][['video_id', 'is_correct']].rename(
        columns={'is_correct': 'base_correct'})
    reason = df[df['question_type'] == reason_type][['video_id', 'is_correct']].rename(
        columns={'is_correct': 'reason_correct'})
    merged = pd.merge(base, reason, on='video_id')
    if len(merged) == 0:
        return 0.0, 0
    both = (merged['base_correct'] & merged['reason_correct']).sum()
    return both / len(merged) * 100, len(merged)

par, par_n = compute_paired_accuracy(df, 'predictive', 'predictive_reason')
car, car_n = compute_paired_accuracy(df, 'counterfactual', 'counterfactual_reason')

acc_d = type_stats['descriptive']['correct'] / max(type_stats['descriptive']['total'], 1) * 100
acc_e = type_stats['explanatory']['correct'] / max(type_stats['explanatory']['total'], 1) * 100
acc_all = (acc_d + acc_e + par + car) / 4

print(f'PAR  (Predictive + Reason):     {par:.2f}%  (n={par_n})')
print(f'CAR  (Counterfactual + Reason): {car:.2f}%  (n={car_n})')
print(f'Acc@D: {acc_d:.2f}%  |  Acc@E: {acc_e:.2f}%')
print(f'Acc@ALL = (D + E + PAR + CAR) / 4 = {acc_all:.2f}%')